# release16_release_gate_bc_weight_screen_v1

Purpose: repair-screen the BC failure isolated by `release16_checkpoint_distillation_bc_readiness_v1`: the `wf_2023_2024` `release_gate_state` slice had only 11 samples and the current BC fit predicted the entire slice as one high-position action.

Governance:
- Canonical baseline: `soxlbacktest_policy_reimplementation_v1`.
- Direct comparison: `release16_checkpoint_distillation_bc_readiness_v1`.
- Role: `diagnose`.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024` train-local target folds.
- Changed layer: `model_selection` / `bc_loss_weighting`.
- External reference: qlib-converted SOXLBacktest buy/sell baseline, release16 distillation target contract, and existing `state_objective_bc` helper.

Hypothesis: if the failure is loss weighting/sampling rather than target semantics, a deterministic release-gate-weighted BC configuration should improve `wf_2023_2024` release-gate accuracy above 0.60 without breaking the other walk-forward fold slice gates. If no configuration passes, the next mechanism should be target enrichment/state-head representation rather than PPO.

Acceptance criteria:
- Recompute the baseline current BC weighting from the previous notebook.
- Screen multiple deterministic BC weighting configurations against all three folds.
- Select only if all required folds pass global and release-gate slice gates.
- Keep `promotion_allowed=false` and `actor_retention_allowed=false`.


In [1]:

from __future__ import annotations

import json
import math
import random
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "SPEC.md").exists() and (candidate / "src/soxl_quant").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    state_objective_bc_sample_weights,
)
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

EXPERIMENT = "release16_release_gate_bc_weight_screen_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
SOURCE_CARD = ROOT / "runtime/model_selection/card/release16_checkpoint_distillation_bc_readiness_v1"
CONTRACT_CARD = ROOT / "runtime/model_selection/card/release16_checkpoint_distillation_ytd_alignment_contract_v1"
RETENTION_GATE = ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json"
TARGET_CONTRACT = CONTRACT_CARD / "distillation_target_contract.csv"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
SEED = 20260615
CARD.mkdir(parents=True, exist_ok=True)

TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=400, lr=0.001, batch_size=128, seed=SEED)
CONFIGS: dict[str, StateObjectiveBCSampleWeightConfig] = {
    "baseline_current": StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=6.0,
        downtrend_multiplier=2.0,
        release_gate_multiplier=16.0,
        objective_weight_power=0.20,
        class_balanced_beta=0.99,
        weight_cap=35.0,
    ),
    "release32_cap50": StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=6.0,
        downtrend_multiplier=2.0,
        release_gate_multiplier=32.0,
        objective_weight_power=0.20,
        class_balanced_beta=0.99,
        weight_cap=50.0,
    ),
    "release64_cap80": StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=6.0,
        downtrend_multiplier=2.0,
        release_gate_multiplier=64.0,
        objective_weight_power=0.20,
        class_balanced_beta=0.99,
        weight_cap=80.0,
    ),
    "release96_cap120": StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=6.0,
        downtrend_multiplier=2.0,
        release_gate_multiplier=96.0,
        objective_weight_power=0.20,
        class_balanced_beta=0.99,
        weight_cap=120.0,
    ),
    "release64_beta995": StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=6.0,
        downtrend_multiplier=2.0,
        release_gate_multiplier=64.0,
        objective_weight_power=0.20,
        class_balanced_beta=0.995,
        weight_cap=80.0,
    ),
    "release64_no_objective": StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=6.0,
        downtrend_multiplier=2.0,
        release_gate_multiplier=64.0,
        objective_weight_power=0.0,
        class_balanced_beta=0.99,
        weight_cap=80.0,
    ),
    "release128_cap160": StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=6.0,
        downtrend_multiplier=2.0,
        release_gate_multiplier=128.0,
        objective_weight_power=0.20,
        class_balanced_beta=0.99,
        weight_cap=160.0,
    ),
    "release64_downtrend4": StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=6.0,
        downtrend_multiplier=4.0,
        release_gate_multiplier=64.0,
        objective_weight_power=0.20,
        class_balanced_beta=0.99,
        weight_cap=100.0,
    ),
}

WALK_FORWARD_WINDOWS = {
    "wf_2020": {
        "target_fold": "wf_2020_train",
        "train": ("2010-03-11", "2018-12-31"),
        "valid": ("2019-01-01", "2019-12-31"),
        "test": ("2020-01-01", "2020-12-31"),
    },
    "wf_2022": {
        "target_fold": "wf_2022_train",
        "train": ("2010-03-11", "2020-12-31"),
        "valid": ("2021-01-01", "2021-12-31"),
        "test": ("2022-01-01", "2022-12-31"),
    },
    "wf_2023_2024": {
        "target_fold": "wf_2023_2024_train",
        "train": ("2010-03-11", "2022-12-31"),
        "valid": ("2023-01-01", "2023-12-31"),
        "test": ("2024-01-01", "2024-12-31"),
    },
}
SLICE_COLS = ("release_gate_state", "is_rebound_cluster", "is_downtrend_state", "drawdown_control_state")
REQUIRED_RULES = {
    "all": {"min_samples": 100, "min_accuracy": 0.55, "required": True},
    "release_gate_state": {"min_samples": 5, "min_accuracy": 0.60, "required": True},
    "is_downtrend_state": {"min_samples": 100, "min_accuracy": 0.65, "required": False},
}


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == object:
        return series.fillna(False).astype(str).str.lower().isin({"true", "1", "yes", "y", "t"})
    return series.fillna(False).astype(bool)


def build_replay_feature_frame(profile):
    dataset = build_dataset(
        train_seg=("2010-03-11", "2022-12-31"),
        valid_seg=("2023-01-01", "2023-12-31"),
        test_seg=("2024-01-01", "2024-12-31"),
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    parts = [segment_frame(dataset, name) for name in ("train", "valid", "test")]
    frame = pd.concat(parts).sort_index()
    return frame.loc[~frame.index.duplicated(keep="last")].copy()


def actor_predictions_and_losses(actor: Any, obs_array: np.ndarray, action_array: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if len(obs_array) == 0:
        return np.asarray([], dtype=np.int64), np.asarray([], dtype=np.float32)
    device = next(actor.parameters()).device
    obs_tensor = torch.as_tensor(obs_array, dtype=torch.float32, device=device)
    label_tensor = torch.as_tensor(action_array, dtype=torch.long, device=device)
    loss_fn = torch.nn.CrossEntropyLoss(reduction="none")
    with torch.no_grad():
        logits = actor(obs_tensor)
        if isinstance(logits, tuple):
            logits = logits[0]
        preds = logits.argmax(-1).detach().cpu().numpy().astype(np.int64)
        losses = loss_fn(logits, label_tensor).detach().cpu().numpy().astype(np.float32)
    return preds, losses


def gate_details(slice_metrics: pd.DataFrame) -> dict[str, Any]:
    indexed = slice_metrics.set_index("slice") if not slice_metrics.empty else pd.DataFrame()
    details: dict[str, Any] = {}
    failed_required: list[str] = []
    for name, rule in REQUIRED_RULES.items():
        if name not in indexed.index:
            row = {"samples": 0, "accuracy": None, "loss": None, "gate": False, **rule}
        else:
            metric = indexed.loc[name]
            samples = int(metric.get("samples", 0))
            accuracy = float(metric.get("accuracy", float("nan")))
            loss = float(metric.get("loss", float("nan")))
            gate = samples >= int(rule["min_samples"]) and math.isfinite(accuracy) and accuracy >= float(rule["min_accuracy"])
            row = {"samples": samples, "accuracy": accuracy, "loss": loss, "gate": bool(gate), **rule}
        details[name] = row
        if bool(rule["required"]) and not bool(row["gate"]):
            failed_required.append(name)
    details["failed_required_slices"] = failed_required
    details["required_gate"] = len(failed_required) == 0
    return details

print("root", ROOT)
print("card", CARD)


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_release_gate_bc_weight_screen_v1


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:

profile = load_profile(PROFILE_NAME)
targets = pd.read_csv(TARGET_CONTRACT)
targets["datetime"] = pd.to_datetime(targets["datetime"]).dt.normalize()
for col in ("training_label_allowed", "is_rebound_cluster", "is_downtrend_state", "release_gate_state", "drawdown_control_state"):
    if col in targets.columns:
        targets[col] = normalize_bool_series(targets[col])

init_qlib(QLIB_DATA_PATH)
frame = build_replay_feature_frame(profile)
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
action_levels = list(action_levels_for_profile(profile))

config_grid = pd.DataFrame([{"config_id": k, **asdict(v)} for k, v in CONFIGS.items()])
config_grid.to_csv(CARD / "config_grid.csv", index=False)
pd.DataFrame({"feature": feature_columns}).to_csv(CARD / "feature_schema.csv", index=False)

fold_targets_by_fold: dict[str, pd.DataFrame] = {}
fold_dataset_by_fold: dict[str, Any] = {}
fold_scope_rows: list[dict[str, Any]] = []
for fold, spec in WALK_FORWARD_WINDOWS.items():
    train_start = pd.Timestamp(spec["train"][0])
    train_end = pd.Timestamp(spec["train"][1])
    valid_start = pd.Timestamp(spec["valid"][0])
    valid_end = pd.Timestamp(spec["valid"][1])
    test_start = pd.Timestamp(spec["test"][0])
    test_end = pd.Timestamp(spec["test"][1])
    fold_all = targets[targets["fold"].astype(str).eq(spec["target_fold"])].copy()
    fold_targets = fold_all[fold_all["datetime"].between(train_start, train_end)].copy()
    fold_targets = fold_targets[fold_targets["training_label_allowed"].fillna(False).astype(bool)].copy()
    valid_overlap = fold_all[fold_all["datetime"].between(valid_start, valid_end)]
    test_overlap = fold_all[fold_all["datetime"].between(test_start, test_end)]
    dataset = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile, sample_size=0, rng_seed=SEED)
    fold_targets_by_fold[fold] = fold_targets
    fold_dataset_by_fold[fold] = dataset
    fold_scope_rows.append({
        "fold": fold,
        "target_fold": spec["target_fold"],
        "target_rows": int(len(fold_targets)),
        "bc_samples": int(len(dataset.actions)),
        "valid_overlap_rows": int(len(valid_overlap)),
        "test_overlap_rows": int(len(test_overlap)),
        "target_scope_ok": bool(len(fold_targets) > 0 and len(test_overlap) == 0),
        "release_gate_rows": int(dataset.metadata.get("release_gate_state", pd.Series(False, index=dataset.metadata.index)).fillna(False).astype(bool).sum()),
    })

fold_scope = pd.DataFrame(fold_scope_rows)
fold_scope.to_csv(CARD / "fold_scope_audit.csv", index=False)
print("profile", profile.name, "features", len(feature_columns), "actions", action_levels)
display(config_grid)
display(fold_scope)


[34487:MainThread](2026-06-15 03:50:30,234) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[34487:MainThread](2026-06-15 03:50:30,633) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[34487:MainThread](2026-06-15 03:50:30,633) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[34487:MainThread](2026-06-15 03:50:31,837) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[34487:MainThread](2026-06-15 03:50:31,837) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[34487:MainThread](2026-06-15 03:50:31,838) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


profile sparse_release_bc_ppo_screen_v1 features 59 actions [np.float32(0.0), np.float32(0.1), np.float32(0.2), np.float32(0.3), np.float32(0.4), np.float32(0.5), np.float32(0.6), np.float32(0.7), np.float32(0.8), np.float32(0.9), np.float32(1.0)]


,config_id,rebound_multiplier,downtrend_multiplier,release_gate_multiplier,ordinary_multiplier,objective_weight_power,class_balanced_beta,weight_floor,weight_cap,normalize_mean
0,baseline_current,6.0,2.0,16.0,1.0,0.2,0.990,0.05,35.0,True
1,release32_cap50,6.0,2.0,32.0,1.0,0.2,0.990,0.05,50.0,True
2,release64_cap80,6.0,2.0,64.0,1.0,0.2,0.990,0.05,80.0,True
3,release96_cap120,6.0,2.0,96.0,1.0,0.2,0.990,0.05,120.0,True
4,release64_beta995,6.0,2.0,64.0,1.0,0.2,0.995,0.05,80.0,True
5,release64_no_objective,6.0,2.0,64.0,1.0,0.0,0.990,0.05,80.0,True
6,release128_cap160,6.0,2.0,128.0,1.0,0.2,0.990,0.05,160.0,True
7,release64_downtrend4,6.0,4.0,64.0,1.0,0.2,0.990,0.05,100.0,True


,fold,target_fold,target_rows,bc_samples,valid_overlap_rows,test_overlap_rows,target_scope_ok,release_gate_rows
0,wf_2020,wf_2020_train,2218,1905,252,0,True,32
1,wf_2022,wf_2022_train,253,253,252,0,True,8
2,wf_2023_2024,wf_2023_2024_train,756,756,0,0,True,11


In [3]:

metric_rows: list[dict[str, Any]] = []
slice_frames: list[pd.DataFrame] = []
action_frames: list[pd.DataFrame] = []
trace_frames: list[pd.DataFrame] = []

for config_id, weight_config in CONFIGS.items():
    print("config", config_id)
    for fold, fold_targets in fold_targets_by_fold.items():
        set_seed(SEED)
        dataset = fold_dataset_by_fold[fold]
        weights = state_objective_bc_sample_weights(dataset.metadata, actions=dataset.actions, config=weight_config)
        result = fit_state_objective_bc_actor_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            train_config=TRAIN_CONFIG,
            sample_weight_config=weight_config,
            sample_size=0,
            rng_seed=SEED,
        )
        slice_metrics = evaluate_bc_actor_slice_fit(
            result.actor,
            result.obs,
            result.actions,
            dataset.metadata,
            slice_cols=SLICE_COLS,
        )
        details = gate_details(slice_metrics)
        tagged = slice_metrics.copy()
        tagged.insert(0, "config_id", config_id)
        tagged.insert(1, "fold", fold)
        slice_frames.append(tagged)

        action_dist = result.action_distribution.copy()
        action_dist.insert(0, "config_id", config_id)
        action_dist.insert(1, "fold", fold)
        if action_levels:
            action_dist["action_level"] = action_dist["action"].map({i: level for i, level in enumerate(action_levels)})
        action_frames.append(action_dist)

        preds, losses = actor_predictions_and_losses(result.actor, result.obs, result.actions)
        trace = dataset.metadata.copy().reset_index(drop=True)
        trace.insert(0, "config_id", config_id)
        trace["fold"] = fold
        trace["label_action"] = result.actions.astype(int)
        trace["predicted_action"] = preds.astype(int)
        trace["correct"] = trace["label_action"].eq(trace["predicted_action"])
        trace["loss"] = losses.astype(float)
        if action_levels:
            trace["label_position_level"] = trace["label_action"].map({i: level for i, level in enumerate(action_levels)})
            trace["predicted_position_level"] = trace["predicted_action"].map({i: level for i, level in enumerate(action_levels)})
        release_trace = trace[trace.get("release_gate_state", pd.Series(False, index=trace.index)).fillna(False).astype(bool)].copy()
        keep_cols = [
            "config_id", "fold", "datetime", "candidate", "action", "teacher_position",
            "rebound_target_position", "risk_target_position", "distillation_target_position",
            "objective_weight", "is_rebound_cluster", "is_downtrend_state", "release_gate_state",
            "drawdown_control_state", "strategy_drawdown", "panic_score", "mom_5", "mom_21",
            "drawdown_63", "rv_21", "vol_of_vol_21", "label_action", "predicted_action",
            "label_position_level", "predicted_position_level", "correct", "loss",
        ]
        trace_frames.append(release_trace[[c for c in keep_cols if c in release_trace.columns]])

        all_detail = details["all"]
        release_detail = details["release_gate_state"]
        downtrend_detail = details.get("is_downtrend_state", {})
        metric_rows.append({
            "config_id": config_id,
            "fold": fold,
            "required_gate": bool(details["required_gate"]),
            "failed_required_slices": ",".join(details["failed_required_slices"]),
            "all_accuracy": all_detail.get("accuracy"),
            "all_loss": all_detail.get("loss"),
            "all_samples": all_detail.get("samples"),
            "release_accuracy": release_detail.get("accuracy"),
            "release_loss": release_detail.get("loss"),
            "release_samples": release_detail.get("samples"),
            "downtrend_accuracy": downtrend_detail.get("accuracy"),
            "downtrend_samples": downtrend_detail.get("samples"),
            "before_accuracy": float(result.metrics["before_accuracy"]),
            "after_accuracy": float(result.metrics["after_accuracy"]),
            "before_loss": float(result.metrics["before_loss"]),
            "after_loss": float(result.metrics["after_loss"]),
            "loss_improvement": float(result.metrics["before_loss"] - result.metrics["after_loss"]),
            "weight_mean": float(weights.mean()) if len(weights) else None,
            "weight_min": float(weights.min()) if len(weights) else None,
            "weight_max": float(weights.max()) if len(weights) else None,
            "weight_p95": float(np.quantile(weights, 0.95)) if len(weights) else None,
        })

config_metrics = pd.DataFrame(metric_rows)
slice_metrics_all = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
action_distribution = pd.concat(action_frames, ignore_index=True) if action_frames else pd.DataFrame()
release_trace = pd.concat(trace_frames, ignore_index=True) if trace_frames else pd.DataFrame()

config_metrics.to_csv(CARD / "config_fold_metrics.csv", index=False)
slice_metrics_all.to_csv(CARD / "config_slice_metrics.csv", index=False)
action_distribution.to_csv(CARD / "config_action_distribution.csv", index=False)
release_trace.to_csv(CARD / "release_gate_prediction_trace.csv", index=False)

display(config_metrics)
display(slice_metrics_all[slice_metrics_all["slice"].eq("release_gate_state")])


config baseline_current


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


config release32_cap50


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


,config_id,fold,required_gate,failed_required_slices,all_accuracy,all_loss,all_samples,release_accuracy,release_loss,release_samples,...,downtrend_samples,before_accuracy,after_accuracy,before_loss,after_loss,loss_improvement,weight_mean,weight_min,weight_max,weight_p95
0,baseline_current,wf_2020,True,,0.724934,1.183302,1905,0.750000,1.147921,32,...,708,0.304987,0.724934,1.605351,1.183302,0.422050,1.0,0.394806,14.723673,1.486623
1,baseline_current,wf_2022,True,,0.873518,1.032363,253,0.875000,1.023031,8,...,118,0.138340,0.873518,1.614201,1.032363,0.581837,1.0,0.351684,6.909615,4.702914
2,baseline_current,wf_2023_2024,False,release_gate_state,0.742063,1.167221,756,0.363636,1.557956,11,...,478,0.074074,0.742063,1.613803,1.167221,0.446582,1.0,0.226098,9.619936,2.185272
3,release32_cap50,wf_2020,True,,0.677690,1.227021,1905,0.812500,1.076646,32,...,708,0.304987,0.677690,1.605351,1.227021,0.378330,1.0,0.341602,18.199345,1.286289
4,release32_cap50,wf_2022,True,,0.806324,1.097241,253,0.875000,1.024265,8,...,118,0.138340,0.806324,1.614201,1.097240,0.516960,1.0,0.318414,8.937080,4.258012
5,release32_cap50,wf_2023_2024,False,release_gate_state,0.734127,1.180102,756,0.363636,1.544351,11,...,478,0.074074,0.734127,1.613803,1.180102,0.433701,1.0,0.204894,12.453912,1.980328
6,release64_cap80,wf_2020,True,,0.622047,1.278385,1905,0.937500,0.975542,32,...,708,0.304987,0.622047,1.605351,1.278385,0.326966,1.0,0.287204,24.481920,1.081454
7,release64_cap80,wf_2022,True,,0.822134,1.084200,253,1.000000,0.906998,8,...,118,0.138340,0.822134,1.614201,1.084200,0.530000,1.0,0.272252,12.226275,3.640705
8,release64_cap80,wf_2023_2024,True,,0.723545,1.186262,756,0.727273,1.178055,11,...,478,0.074074,0.723545,1.613803,1.186262,0.427541,1.0,0.184062,17.900362,1.778988
9,release96_cap120,wf_2020,True,,0.633596,1.275510,1905,0.937500,0.972552,32,...,708,0.304987,0.633596,1.605351,1.275510,0.329842,1.0,0.238221,30.459686,0.897009


,config_id,fold,slice,samples,accuracy,loss,label_unique_actions,predicted_unique_actions,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share
4,baseline_current,wf_2020,release_gate_state,32,0.750000,1.147921,3,3,4,0.843750,4,0.718750
9,baseline_current,wf_2022,release_gate_state,8,0.875000,1.023031,4,3,4,0.375000,4,0.375000
14,baseline_current,wf_2023_2024,release_gate_state,11,0.363636,1.557956,4,1,3,0.454545,4,1.000000
19,release32_cap50,wf_2020,release_gate_state,32,0.812500,1.076646,3,3,4,0.843750,4,0.781250
24,release32_cap50,wf_2022,release_gate_state,8,0.875000,1.024265,4,3,4,0.375000,4,0.375000
29,release32_cap50,wf_2023_2024,release_gate_state,11,0.363636,1.544351,4,1,3,0.454545,4,1.000000
34,release64_cap80,wf_2020,release_gate_state,32,0.937500,0.975542,3,3,4,0.843750,4,0.781250
39,release64_cap80,wf_2022,release_gate_state,8,1.000000,0.906998,4,4,4,0.375000,4,0.375000
44,release64_cap80,wf_2023_2024,release_gate_state,11,0.727273,1.178055,4,2,3,0.454545,4,0.545455
49,release96_cap120,wf_2020,release_gate_state,32,0.937500,0.972552,3,3,4,0.843750,4,0.781250


In [4]:

summary_rows: list[dict[str, Any]] = []
for config_id, group in config_metrics.groupby("config_id", sort=False):
    folds_present = set(group["fold"].astype(str))
    all_folds = set(WALK_FORWARD_WINDOWS)
    all_required_gate = bool(folds_present == all_folds and group["required_gate"].fillna(False).astype(bool).all())
    min_release_acc = float(pd.to_numeric(group["release_accuracy"], errors="coerce").min())
    min_all_acc = float(pd.to_numeric(group["all_accuracy"], errors="coerce").min())
    avg_after_loss = float(pd.to_numeric(group["after_loss"], errors="coerce").mean())
    release_fail_count = int((pd.to_numeric(group["release_accuracy"], errors="coerce") < REQUIRED_RULES["release_gate_state"]["min_accuracy"]).sum())
    required_fail_count = int((~group["required_gate"].fillna(False).astype(bool)).sum())
    score = (10000.0 if all_required_gate else 0.0) + min_release_acc * 100.0 + min_all_acc * 10.0 - avg_after_loss - required_fail_count * 100.0
    summary_rows.append({
        "config_id": config_id,
        "all_required_gate": all_required_gate,
        "required_fail_count": required_fail_count,
        "release_fail_count": release_fail_count,
        "min_release_accuracy": min_release_acc,
        "min_all_accuracy": min_all_acc,
        "avg_after_loss": avg_after_loss,
        "score": score,
    })

config_summary = pd.DataFrame(summary_rows).sort_values(["all_required_gate", "score"], ascending=[False, False]).reset_index(drop=True)
selected_config = str(config_summary.iloc[0]["config_id"])
selected_pass = bool(config_summary.iloc[0]["all_required_gate"])
baseline_summary = config_summary[config_summary["config_id"].eq("baseline_current")].iloc[0].to_dict()
selected_summary = config_summary.iloc[0].to_dict()
selected_metrics = config_metrics[config_metrics["config_id"].eq(selected_config)].copy()
selected_release_trace = release_trace[release_trace["config_id"].isin(["baseline_current", selected_config])].copy()

config_summary.to_csv(CARD / "config_summary.csv", index=False)
selected_metrics.to_csv(CARD / "selected_config_fold_metrics.csv", index=False)
selected_release_trace.to_csv(CARD / "selected_release_gate_prediction_trace.csv", index=False)

decision = (
    "release_gate_bc_weighting_ready_no_promotion"
    if selected_pass and selected_config != "baseline_current"
    else "release_gate_bc_weighting_not_sufficient_no_promotion"
)
if selected_pass and selected_config == "baseline_current":
    decision = "baseline_release_gate_fit_recomputed_ready_no_promotion"
next_mechanism = (
    "wire_selected_bc_weighting_into_training_handoff_then_run_official_bc_walkforward_replay"
    if decision in {"release_gate_bc_weighting_ready_no_promotion", "baseline_release_gate_fit_recomputed_ready_no_promotion"}
    else "target_enrichment_or_explicit_release_state_head_before_ppo"
)

source_summary = load_json(SOURCE_CARD / "summary.json")
retention_summary = load_json(RETENTION_GATE).get("summary", {})
selection = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "next_mechanism": next_mechanism,
    "selected_config": selected_config,
    "selected_pass": selected_pass,
    "selected_summary": selected_summary,
    "baseline_summary": baseline_summary,
    "profile": PROFILE_NAME,
    "source_decision": source_summary.get("decision"),
    "promotion_allowed": False,
    "actor_retention_allowed": False,
}
write_json(CARD / "selection.json", selection)

summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "role": "diagnose_bc_loss_weighting",
    "canonical_baseline": "soxlbacktest_policy_reimplementation_v1",
    "direct_comparison_baseline": "release16_checkpoint_distillation_bc_readiness_v1",
    "target_window": list(WALK_FORWARD_WINDOWS),
    "changed_layer": ["model_selection", "bc_loss_weighting"],
    "external_reference": "converted SOXLBacktest qlib baseline plus release16 fold-local distillation contract",
    "hypothesis": "The failed wf_2023_2024 release gate can be fixed by deterministic release-gate weighted BC if the target semantics and features are learnable.",
    "acceptance_criteria": [
        "baseline current weighting is recomputed",
        "all three walk-forward folds are screened",
        "selected config passes all required global and release-gate slice gates",
        "no actor is retained or promoted from this diagnostic screen",
    ],
    "expected_tradeoff": "Higher release-gate weight may improve sparse release labels but could degrade ordinary/downtrend fit; select only if required fold gates stay intact.",
    "failure_mode": "If all weighting configs still fail wf_2023_2024 release-gate accuracy, the blocker is target enrichment or representation/state-head rather than PPO length.",
    "comparison_table_row_key": EXPERIMENT,
    "closest_historical_success_case": source_summary.get("closest_historical_success_case", "release16 BC initialization passed wf_2022 candidate gate"),
    "closest_historical_failure_case": "release16_checkpoint_distillation_bc_readiness_v1 failed wf_2023_2024 release_gate_state with 0.3636 accuracy and single-action prediction collapse.",
    "why_not_duplicate": "This screen changes only BC loss weighting over the same fold-local contract; it does not rebuild the contract or rerun PPO.",
    "profile": PROFILE_NAME,
    "train_config": asdict(TRAIN_CONFIG),
    "selected_config": selected_config,
    "selected_pass": selected_pass,
    "selected_summary": selected_summary,
    "baseline_summary": baseline_summary,
    "config_summary": config_summary.to_dict("records"),
    "fold_scope_audit": fold_scope.to_dict("records"),
    "retention_gate_current": {
        "passing_profile_count": retention_summary.get("passing_profile_count"),
        "reference_profile": retention_summary.get("reference_profile"),
        "reference_return_pct": retention_summary.get("reference_return_pct"),
        "current_best_return_pct": retention_summary.get("current_best_return_pct"),
        "required_best_return_pct": retention_summary.get("required_best_return_pct"),
        "max_walkforward_drawdown_pct": retention_summary.get("max_walkforward_drawdown_pct"),
    },
    "next_mechanism": next_mechanism,
    "output_paths": {
        "config_grid": str((CARD / "config_grid.csv").relative_to(ROOT)),
        "fold_scope_audit": str((CARD / "fold_scope_audit.csv").relative_to(ROOT)),
        "config_fold_metrics": str((CARD / "config_fold_metrics.csv").relative_to(ROOT)),
        "config_slice_metrics": str((CARD / "config_slice_metrics.csv").relative_to(ROOT)),
        "config_summary": str((CARD / "config_summary.csv").relative_to(ROOT)),
        "release_gate_prediction_trace": str((CARD / "release_gate_prediction_trace.csv").relative_to(ROOT)),
        "selected_release_gate_prediction_trace": str((CARD / "selected_release_gate_prediction_trace.csv").relative_to(ROOT)),
        "selection": str((CARD / "selection.json").relative_to(ROOT)),
        "summary": str((CARD / "summary.json").relative_to(ROOT)),
    },
    "conclusion": (
        "A deterministic BC weighting config fixes the release-gate slice screen; next step is official BC walk-forward replay before any PPO or actor retention."
        if decision in {"release_gate_bc_weighting_ready_no_promotion", "baseline_release_gate_fit_recomputed_ready_no_promotion"}
        else "Release-gate weighting alone is not sufficient; move to target enrichment or explicit release state-head before PPO."
    ),
}
write_json(CARD / "summary.json", summary)

print("selected", selected_config, "decision", decision)
display(config_summary)
display(selected_metrics)


selected release64_no_objective decision release_gate_bc_weighting_ready_no_promotion


,config_id,all_required_gate,required_fail_count,release_fail_count,min_release_accuracy,min_all_accuracy,avg_after_loss,score
0,release64_no_objective,True,0,0,0.818182,0.652493,1.198374,10087.144742
1,release64_downtrend4,True,0,0,0.818182,0.632021,1.186891,10086.951501
2,release128_cap160,True,0,0,0.818182,0.575853,1.250198,10086.326514
3,release64_beta995,True,0,0,0.727273,0.663517,1.196344,10078.166099
4,release96_cap120,True,0,0,0.727273,0.633596,1.232191,10077.831040
5,release64_cap80,True,0,0,0.727273,0.622047,1.182949,10077.764796
6,baseline_current,False,1,1,0.363636,0.724934,1.127629,-57.514648
7,release32_cap50,False,1,1,0.363636,0.677690,1.168121,-58.027582


,config_id,fold,required_gate,failed_required_slices,all_accuracy,all_loss,all_samples,release_accuracy,release_loss,release_samples,...,downtrend_samples,before_accuracy,after_accuracy,before_loss,after_loss,loss_improvement,weight_mean,weight_min,weight_max,weight_p95
15,release64_no_objective,wf_2020,True,,0.652493,1.248661,1905,0.906250,0.999740,32,...,708,0.304987,0.652493,1.605351,1.248661,0.356690,1.0,0.395250,33.692024,0.797068
16,release64_no_objective,wf_2022,True,,0.754941,1.149593,253,1.000000,0.904931,8,...,118,0.138340,0.754941,1.614201,1.149593,0.464607,1.0,0.175958,23.341574,2.353012
17,release64_no_objective,wf_2023_2024,True,,0.707672,1.196867,756,0.818182,1.088289,11,...,478,0.074074,0.707672,1.613803,1.196868,0.416936,1.0,0.224537,21.836599,1.431787


## Conclusion

The final cell renders `summary.json`. This diagnostic does not retain an actor and does not change runtime model status.


In [5]:

summary = load_json(CARD / "summary.json")
lines = [
    f"Decision: `{summary['decision']}`.",
    f"Selected config: `{summary['selected_config']}`; selected pass: `{summary['selected_pass']}`.",
    f"Baseline min release accuracy: `{summary['baseline_summary']['min_release_accuracy']}`; selected min release accuracy: `{summary['selected_summary']['min_release_accuracy']}`.",
    f"Next mechanism: `{summary['next_mechanism']}`.",
    summary["conclusion"],
]
display(Markdown("\n".join(f"- {line}" for line in lines)))


- Decision: `release_gate_bc_weighting_ready_no_promotion`.
- Selected config: `release64_no_objective`; selected pass: `True`.
- Baseline min release accuracy: `0.36363636363636365`; selected min release accuracy: `0.8181818181818182`.
- Next mechanism: `wire_selected_bc_weighting_into_training_handoff_then_run_official_bc_walkforward_replay`.
- A deterministic BC weighting config fixes the release-gate slice screen; next step is official BC walk-forward replay before any PPO or actor retention.